# 2. Diet problem (Week 2 tutorial Q6a; the slides call it Q5a): minimise cost

Lecture: *Week 3 - Linear Programming 2*, PDF pages 7–11 (*Diet Problem (Tutorial Question 5a)* and
*5a Solution via Pyomo (Problem Based)*).

**The problem.** Choose amounts $x_1, \dots, x_{10} \ge 0$ of 10 foods so that the diet contains at least
100 units of vitamin A, 120 of B, 200 of C and 80 of D, and at most 300 units of fat, at the lowest cost:

minimise $\sum_i c_i x_i$ subject to $\sum_i v^A_i x_i \ge 100$, $\sum_i v^B_i x_i \ge 120$,
$\sum_i v^C_i x_i \ge 200$, $\sum_i v^D_i x_i \ge 80$, $\sum_i f_i x_i \le 300$, $x_i \ge 0$.

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** Minimum cost **93.25**, using foods 2, 5 and 10 (`x[1]` = 9.09, `x[4]` = 1.43,
`x[9]` = 2.27): the same answer as Week 02's `linprog` notebook for Q6a.

*Original script: `5 Source Files/Python/Week2_question_5a_pyomo.py`. Runs in the browser with HiGHS instead of GLPK; the model and answers are unchanged.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

Load Pyomo's modelling tools.

In [ ]:
from pyomo.environ import *

The data from the slide's table, one list per column. Position `i` in each list is food `i + 1` (Python counts from 0), and a '-' in the table is 0.

In [ ]:
# Define the cost, vitamin, and fat values as lists
c = [10, 6, 8, 16, 8, 3, 6, 8, 12, 12]
vita = [18, 8, 22, 6, 0, 0, 8, 4, 0, 12]
vitb = [4, 6, 10, 12, 14, 0, 0, 0, 6, 20]
vitc = [0, 22, 0, 0, 0, 20, 0, 8, 10, 0]
vitd = [6, 14, 0, 6, 10, 8, 6, 0, 12, 4]
fat = [0, 0, 8, 0, 0, 6, 20, 40, 0, 0]

no_vars = len(c)

A *concrete model* is the container for the LP.

In [ ]:
# Create a concrete model
model = ConcreteModel()

Ten non-negative variables: `x[i]` is the amount of food `i + 1`.

In [ ]:
# Define the decision variables
model.x = Var(range(no_vars), domain=NonNegativeReals)

The objective: the total cost $\sum_i c_i x_i$, minimised.

In [ ]:
# Define the objective function
model.obj = Objective(expr=sum(c[i] * model.x[i] for i in range(no_vars)), sense=minimize)

One constraint per row of the right-hand table: four vitamin minimums and one fat maximum. `sum(vita[i] * model.x[i] for i in range(no_vars))` is $\sum_i v^A_i x_i$.

In [ ]:
# Define constraints
model.vita_min = Constraint(expr=sum(vita[i] * model.x[i] for i in range(no_vars)) >= 100)  # vitamin A: at least 100
model.vitb_min = Constraint(expr=sum(vitb[i] * model.x[i] for i in range(no_vars)) >= 120)  # vitamin B: at least 120
model.vitc_min = Constraint(expr=sum(vitc[i] * model.x[i] for i in range(no_vars)) >= 200)  # vitamin C: at least 200
model.vitd_min = Constraint(expr=sum(vitd[i] * model.x[i] for i in range(no_vars)) >= 80)  # vitamin D: at least 80
model.fat_max = Constraint(expr=sum(fat[i] * model.x[i] for i in range(no_vars)) <= 300)  # fat: at most 300

Solve with HiGHS and display the solution. Under *Constraints*, a *Body* equal to its bound means the constraint is binding.

In [ ]:
# Solve the optimization problem
solver = SolverFactory('appsi_highs')
solver.solve(model, tee=False)  # Set tee=True to display solver output

model.display()

 

## What do you notice? / Try it

1. Which constraints are binding? 2. Change the vitamin C minimum from 200 to 150 and re-run: what happens to
the cost?

<details><summary>Answers</summary>

1. Vitamins A, B and C (their *Body* equals the minimum; values like `120.00000000000001` are rounding).
   Vitamin D (about 150.6) and fat (0) are not binding: this cheapest diet contains no fat at all.
2. The cost falls to about **88.27**. Relaxing a binding constraint can only help (or leave the cost
   unchanged); relaxing vitamin D's minimum would change nothing, because it is not binding.
</details>